# RSW / 02 — Hawkes vs Markov for MNQ 5-minute extremes: decay and next-bar forecast (2020-01 – 2023-06)

Protocol: [`RSW 003`](../../discussions/MNQ_Regime_Switching/003_claude_rsw02_hawkes_pot_preregistration.md). The event is |z_t| ≥ u, with u the 0.96 quantile of |z| on the fit window. z is the RSW 01 series.

- **H1 (decay)**: X = Σ_{k=6..60} [C(k) − C_null(k)] has a 95% lower bound > 0 against the binary chain and the simulated RSW M1 (2-state) and M2 (3-state) models. H1 is necessary, not sufficient: volatility persistence left in z also passes it (Markov smoke run).
- **H2 (forecast)**: HX, HXM or HP beats MC, M1 and M2 on the out-of-sample Bernoulli log score (Holm across 9 tests ≤ 0.05).
- **H3 (stable)**: the winner's branching ratio n is in (0.05, 0.95) in every fit, with a range ≤ 0.15.

In [ ]:
import json, os, sys, tempfile, pickle
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("RSW_SYNTHETIC") == "1"
SYNTH_HAWKES = os.environ.get("RSW_SYNTH_HAWKES") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("RSW_SYNTH_DIR") or tempfile.mkdtemp(prefix="rsw_")
    os.environ["RSW_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V
from equity_trading import regime_ar as R
from equity_trading import hawkes as H
from equity_trading import micro_bars as MB

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
RC, HC = R.RSWConfig(), H.HWKConfig()
D = dt.date
OUT = Path("02"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", H.HWK_VERSION, R.RSW_VERSION, P.MRP_VERSION, "| synthetic" + (" hawkes" if SYNTH_HAWKES else " markov") if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    if SYNTH_HAWKES:
        H.write_synthetic_bursts(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.train[1], CFG)
    else:
        R.write_synthetic_regimes(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.train[1], CFG)

In [ ]:
if SYNTHETIC:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2020, 7, 1)), CFG.refits_train, CFG.train[1]
else:
    FIRST_FIT, REFITS, END = (CFG.train[0], D(2021, 1, 1)), CFG.refits_train, CFG.train[1]
grid = P.load_grid(CFG.warmup_start, END, CFG)
sig2 = V.event_sigma(grid, FIRST_FIT, REFITS, END, CFG)
m1 = MB.load_1m(CFG.root, CFG.warmup_start - dt.timedelta(days=3), END + dt.timedelta(days=1))
S = R.build_series(grid, sig2, m1, CFG, RC)
S["z0"] = S["ret"] / np.sqrt(sig2.reindex(S.index).to_numpy(float))   # without the day rescale (sensitivity)
S = S[(S["trade_date"] >= CFG.train[0]) & (S["trade_date"] <= END)]
del m1, grid
td = S["trade_date"].to_numpy()
obs = S["obs"].to_numpy(bool)
ALL = np.ones(len(S), bool)
u_all = H.threshold(S, ALL, HC.q_main)
e_all, y_all = H.events(S, u_all)
print(f"{len(S):,} bars | observed {obs.sum():,} | u(0.96) = {u_all:.3f} | events {int(e_all.sum()):,} "
      f"({e_all[obs].mean():.4f} of observed bars)")
ev = pd.DataFrame({"e": e_all[obs], "slot": S["slot"].to_numpy()[obs], "session": S["session"].to_numpy()[obs],
                   "year": pd.to_datetime(S["trade_date"]).dt.year.to_numpy()[obs]})
display(pd.concat({"by session": ev.groupby("session")["e"].mean(), "by year": ev.groupby("year")["e"].mean()}).round(4).to_frame("event rate").T)
fig, ax = plt.subplots(figsize=(12, 2.8))
ev.groupby("slot")["e"].mean().plot(ax=ax, lw=0.8); ax.axhline(e_all[obs].mean(), color="grey", lw=0.6)
ax.set_title("event rate by slot (z is deseasonalized: should be flat)"); plt.tight_layout(); plt.show()

## 1. Test 1: decay of clustering against the Markov nulls

In [ ]:
t0 = dt.datetime.now()
FULL = {K: R.fit(S, K, False, ALL, RC) for K in (2, 3)}   # descriptive refits on the whole window (the HMM nulls)
for K, m in FULL.items():
    print(f"M{K - 1} on the whole window"); display(m.table().round(4))
def nulls_for(fits, u):
    return {f"M{K - 1}": H.hmm_null_curve(m, u, HC.hmm_sim_bars, HC.kmax, seed=HC.seed) for K, m in fits.items()}
DEC = {}
for q in (HC.q_main,) + HC.q_sens:
    u = H.threshold(S, ALL, q)
    DEC[("z", q)] = H.decay_test(S, u, nulls_for(FULL, u), HC)
# sensitivity: z without the day rescale (HMM nulls refitted on z0)
S0 = S.assign(z=S["z0"], zl=np.where(S["has_lag"], np.r_[np.nan, S["z0"].to_numpy()[:-1]], 0.0))
u0 = H.threshold(S0, ALL, HC.q_main)
DEC[("z0", HC.q_main)] = H.decay_test(S0, u0, nulls_for({K: R.fit(S0, K, False, ALL, RC) for K in (2, 3)}, u0), HC)
dec_tab = pd.DataFrame({f"{c} q{q}": {k: v for k, v in d.items() if np.isscalar(v)} for (c, q), d in DEC.items()})
display(dec_tab)
print("elapsed", dt.datetime.now() - t0)

In [ ]:
d = DEC[("z", HC.q_main)]
k = np.arange(HC.kmax + 1)
fig, ax = plt.subplots(1, 2, figsize=(15, 4))
for a, logy in zip(ax, (False, True)):
    a.plot(k[1:], (d["C"] - d["p"])[1:], label="observed C(k) - p", color="k", lw=1)
    a.plot(k[1:], (d["C_chain"] - d["p"])[1:], label="binary chain", lw=1)
    a.plot(k[1:], (d["C_M1"] - d["p"])[1:], label="RSW M1, 2 states (simulated)", lw=1)
    a.plot(k[1:], (d["C_M2"] - d["p"])[1:], label="RSW M2, 3 states (simulated)", lw=1)
    a.axvspan(*HC.excess_lags, color="grey", alpha=0.12)
    a.set_xlabel("lag k (5-minute bars)"); a.legend(fontsize=8)
    if logy:
        a.set_yscale("symlog", linthresh=1e-3); a.set_title("same, symlog")
    else:
        a.set_title(f"excess P(event at t+k | event at t) over base p = {d['p']:.4f}")
plt.tight_layout(); plt.savefig(OUT / "decay.png", dpi=110); plt.show()
H1 = d["H1"]
print("H1:", H1, "|", " | ".join(f"X vs {n} {d[f'excess_vs_{n}']:.4f} (lo95 {d[f'lo95_vs_{n}']:.4f})" for n in ("chain", "M1", "M2")))

## 2. Test 2: walk-forward next-bar forecast of an exceedance

In [ ]:
KINDS = ("HX", "HXM", "HP")
cache = OUT / f"fits_{H.HWK_VERSION}{'_synth' if SYNTHETIC else ''}{'_hawkes' if SYNTH_HAWKES else ''}.pkl"
MSAR_FITS = {}
def walk(q):
    edges = list(REFITS) + [END + dt.timedelta(days=1)]
    out = {"e": [], "y": [], "u": [], "day": [], "idx": [], **{m: [] for m in ("B0", "MC", "M1", "M2") + KINDS},
           **{f"{kd}_ratio": [] for kd in KINDS}}
    fits = []
    for a, b in zip(edges[:-1], edges[1:]):
        fm = td < a
        u = H.threshold(S, fm, q)
        e, y = H.events(S, u)
        seg = (td >= a) & (td < b) & obs
        row = {"fit_before": str(a), "u": u}
        out["B0"].append(H.prob_constant(e, obs, fm)[seg])
        pc, chain = H.prob_chain(e, obs, fm); out["MC"].append(pc[seg]); row["chain"] = chain
        for K in (2, 3):
            if (a, K) not in MSAR_FITS:                 # the regime fits do not depend on the threshold
                MSAR_FITS[(a, K)] = R.fit(S, K, False, fm, RC)
            m = MSAR_FITS[(a, K)]
            out[f"M{K - 1}"].append(H.prob_msar(S, m, u)[seg]); row[f"M{K - 1}_phi"] = m.phi.tolist()
        for kd in KINDS:
            f = H.fit_hawkes(kd, e[fm], y[fm], obs[fm], u, HC)
            lam, exc = H.intensity(f, e, y)
            out[kd].append((1 - np.exp(-lam))[seg]); row[kd] = f
            out[f"{kd}_ratio"].append((lam / f.mu)[seg])
        out["e"].append(e[seg]); out["y"].append(y[seg]); out["u"].append(np.full(seg.sum(), u))
        out["day"].append(td[seg]); out["idx"].append(S.index[seg])
        fits.append(row)
        print(f"q {q} fit before {a}: u {u:.3f}, events in fit {int(e[fm & obs].sum())}, "
              + ", ".join(f"{kd} n={row[kd].n:.3f}" for kd in KINDS))
    return {k: (np.concatenate(v) if k != "idx" else np.concatenate([np.asarray(x) for x in v])) for k, v in out.items()}, fits
if cache.exists():
    WF = pickle.load(open(cache, "rb")); print("loaded", cache)
else:
    t0 = dt.datetime.now()
    WF = {q: walk(q) for q in (HC.q_main,) + HC.q_sens}
    pickle.dump(WF, open(cache, "wb")); print("elapsed", dt.datetime.now() - t0)
FC, FITS = WF[HC.q_main]
par = pd.DataFrame([{"fit_before": r["fit_before"], **r[kd].row()} for r in FITS for kd in KINDS])
display(par.round(4))
display(pd.DataFrame([{"fit_before": r["fit_before"], "u": r["u"], **r["chain"], "M1_phi": np.round(r["M1_phi"], 3),
                       "M2_phi": np.round(r["M2_phi"], 3)} for r in FITS]))

In [ ]:
def score_table(fc):
    e, day = fc["e"], fc["day"]
    sc = {m: H.bernoulli_score(fc[m], e) for m in ("B0", "MC", "M1", "M2") + KINDS}
    rows, pv = [], {}
    for kd in KINDS:
        row = {"model": kd, "score_per_bar": sc[kd].mean()}
        for bm in ("B0", "MC", "M1", "M2"):
            t = H.score_dm(sc[kd], sc[bm], day, HC.nw_lags)
            row[f"gain_vs_{bm}"], row[f"t_vs_{bm}"], row[f"p_vs_{bm}"] = t["per_bar_diff"], t["t"], t["p"]
            if bm != "B0":
                pv[(kd, bm)] = t["p"]
        rows.append(row)
    T = pd.DataFrame(rows).set_index("model")
    hp = R.holm({f"{a}|{b}": v for (a, b), v in pv.items()})
    for kd in KINDS:
        for bm in ("MC", "M1", "M2"):
            T.loc[kd, f"p_holm_vs_{bm}"] = hp[f"{kd}|{bm}"]
    T["passes_H2"] = np.logical_and.reduce([(T[f"gain_vs_{bm}"] > 0) & (T[f"p_holm_vs_{bm}"] <= HC.alpha)
                                            for bm in ("MC", "M1", "M2")])
    bench = pd.DataFrame({m: {"score_per_bar": sc[m].mean()} for m in ("B0", "MC", "M1", "M2")}).T
    return T, bench, sc
T, bench, SC = score_table(FC)
display(bench.round(6)); display(T.T)
H2 = bool(T["passes_H2"].any())
WIN = T.loc[T["passes_H2"], "score_per_bar"].idxmax() if H2 else None
print("H2:", H2, "| winner:", WIN)
for q in HC.q_sens:
    Tq, bq, _ = score_table(WF[q][0])
    print(f"--- sensitivity u(q={q}): events OOS {int(WF[q][0]['e'].sum())}")
    display(pd.concat([bq["score_per_bar"], Tq["score_per_bar"]]).to_frame(f"score q{q}").T)
    display(Tq[["gain_vs_MC", "gain_vs_M1", "gain_vs_M2", "p_holm_vs_MC", "p_holm_vs_M1", "p_holm_vs_M2", "passes_H2"]])

In [ ]:
ref = WIN or "HX"
ns = [r[ref].n for r in FITS]
H3 = bool(H2 and all(HC.stab_lo < n_ < HC.stab_hi for n_ in ns) and (max(ns) - min(ns) <= HC.stab_range))
print(f"H3: {H3} | {ref} branching ratio per fit {np.round(ns, 3)} | half-life / median lag per fit",
      [round(r[ref].half_life_bars, 1) for r in FITS])
e = FC["e"]
for m in ("MC", "M1", "M2", ref):
    print(f"--- {m}: lift in the top {HC.lift_top:.0%} of predicted probability", {k: round(v, 4) for k, v in H.lift(FC[m], e, HC.lift_top).items()})
cal = pd.concat({m: H.calibration(FC[m], e) for m in ("MC", "M1", "M2", ref)}, axis=1)
display(cal.round(4))
fig, ax = plt.subplots(figsize=(5, 4))
for m in ("MC", "M1", "M2", ref):
    c = H.calibration(FC[m], e); ax.plot(c["mean_p"], c["rate"], marker="o", label=m)
lim = max(cal.xs("rate", axis=1, level=1).max().max(), 0.01) * 1.05
ax.plot([0, lim], [0, lim], color="grey", lw=0.6); ax.set_xlabel("predicted"); ax.set_ylabel("observed"); ax.legend()
ax.set_title("out-of-sample calibration (deciles)"); plt.tight_layout(); plt.savefig(OUT / "calibration.png", dpi=110); plt.show()

## 3. Sizes: GPD of the exceedances, and whether they grow with excitation (κ)

In [ ]:
edges = list(REFITS) + [END + dt.timedelta(days=1)]
gp = []
for r, a in zip(FITS, edges[:-1]):
    fm = (td < a)
    u = r["u"]; e_, y_ = H.events(S, u)
    lam, _ = H.intensity(r[ref], e_, y_)
    sel = fm & (e_ > 0)
    gp.append({"fit_before": r["fit_before"], "u": u, **H.gpd_fit(y_[sel] - u, lam[sel] / r[ref].mu)})
gp = pd.DataFrame(gp); display(gp.round(4))
sel = FC["e"] > 0
g_oos = H.gpd_fit((FC["y"] - FC["u"])[sel], FC[f"{ref}_ratio"][sel])
print(f"out of sample ({ref} intensity):", {k: round(v, 4) for k, v in g_oos.items()})

In [ ]:
# descriptive: sign persistence of consecutive exceedances within 12 bars (direction is not tested)
zz = S["z"].to_numpy(float); ii = np.flatnonzero(e_all > 0)
gap = np.diff(ii); same = np.sign(zz[ii[1:]]) == np.sign(zz[ii[:-1]])
near = gap <= 12
print(f"pairs within 12 bars: {near.sum():,} | same sign {same[near].mean():.3f} | pairs further apart: same sign {same[~near].mean():.3f}"
      f" | share of exceedances that are negative {np.mean(zz[ii] < 0):.3f}")

## 4. Verdict

In [ ]:
verdict = ("HAWKES (burst timing forecastable beyond Markov; next: use pre-registration)" if (H1 and H2 and H3) else
           "SLOW_CLUSTERING_NO_FORECAST_GAIN (Hawkes or leftover volatility persistence: not identified)" if (H1 and not H2) else
           "SHORT_RUN_HAWKES_FORECAST_ONLY" if (H2 and not H1) else
           "HAWKES_BUT_UNSTABLE" if (H1 and H2) else "MARKOV_SUFFICES")
print("H1", H1, "| H2", H2, "| H3", H3, "|", verdict)
summary = {"study": "RSW 02", "engine": H.HWK_VERSION, "synthetic": SYNTHETIC, "synthetic_hawkes": SYNTH_HAWKES,
           "u_full": u_all, "events_full": int(e_all.sum()),
           "decay": {f"{c}_q{q}": {k: v for k, v in d_.items() if np.isscalar(v)} for (c, q), d_ in DEC.items()},
           "decay_curve": {k_: DEC[("z", HC.q_main)][k_].tolist() for k_ in ("C", "C_chain", "C_M1", "C_M2")},
           "params": json.loads(par.to_json(orient="records")),
           "scores_benchmarks": bench["score_per_bar"].to_dict(), "scores": json.loads(T.to_json(orient="index")),
           "winner": WIN, "branching_ratio_per_fit": ns, "gpd": json.loads(gp.to_json(orient="records")),
           "gpd_oos": g_oos,
           "lift": {m: H.lift(FC[m], FC["e"], HC.lift_top) for m in ("MC", "M1", "M2", ref)},
           "H1": H1, "H2": H2, "H3": H3, "verdict": verdict}
json.dump(summary, open(OUT / "summary.json", "w"), indent=1, default=str)
print("written", OUT / "summary.json")